# 머신러닝 특성 공학(Feature Engineering) 파이프라인 — Telco Customer Churn

**과목:** 빅데이터분석 01분반  |  **학과:** 컴퓨터공학과  |  **학번:** 2204774  |  **이름:** 박소현

---

## 과제 개요
실제 공개 데이터(Telco Customer Churn)를 사용하여 EDA → 결측치 처리 → 인코딩 → 스케일링 → 파생변수 → 변수선택 → 모델 학습/평가까지의 전체 ML 파이프라인을 구현하고, 전처리·변수선택 전략별 성능 차이를 비교·분석한다.

**목표:** 고객 이탈(Churn) 여부 예측 (이진 분류)

### 이 노트북의 구성
- STEP 01 데이터 준비 (자동 다운로드 / shape / 컬럼 설명)
- STEP 02 EDA (결측치, 이상치, 분포, 상관, 타겟분포 / Histogram·Boxplot·Heatmap·Barplot)
- STEP 03 Feature Engineering (결측치·인코딩·스케일링 비교 + 파생변수 5종)
- STEP 04 변수 선택 (제거 전/후 성능 비교)
- STEP 05 모델 학습/평가 (LogReg·RF·XGBoost·LightGBM, 전 지표)
- 실험 비교표 (Base / Exp-1 / Exp-2 / Exp-3)
- 가산점: Pipeline 객체, GridSearchCV, SHAP, Feature Importance 고도화, AutoML 비교
- 마지막 셀에서 **한국어 PDF 보고서 자동 생성**

> Colab에서 위에서 아래로 전부 실행(Runtime → Run all)하면 됩니다. 마지막 셀이 PDF를 만들어 자동 다운로드합니다. (인터넷에서 데이터를 받아오므로 내 컴퓨터 저장공간은 사용하지 않습니다.)

## 0. 환경 설정 (라이브러리 설치 및 import)

In [ ]:
# 필요한 패키지 설치 (Colab 기준). 이미 설치된 경우 빠르게 통과합니다.
!pip install -q reportlab shap xgboost lightgbm
# 한국어 폰트(그래프에 한글이 들어갈 경우 대비) - 실패해도 노트북은 정상 동작
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

FIGDIR = 'figs'
os.makedirs(FIGDIR, exist_ok=True)

# 한글 폰트가 있으면 등록 (없으면 무시)
try:
    import matplotlib.font_manager as fm
    nf = [f for f in fm.findSystemFonts() if 'nanum' in f.lower()]
    if nf:
        for fp in nf:
            fm.fontManager.addfont(fp)
        plt.rcParams['font.family'] = 'NanumGothic'
        print('Korean font registered: NanumGothic')
    else:
        print('Nanum font not found, using default (plot labels are in English)')
except Exception as e:
    print('font setup skipped:', e)

REPORT = {}   # PDF 보고서에 넣을 핵심 수치 저장용
print('setup done')

## STEP 01. 데이터 준비
공개 데이터셋을 URL에서 자동 다운로드한다(별도 업로드 불필요). 타겟 변수는 `Churn`(이탈 여부).

In [ ]:
# 여러 미러를 순서대로 시도 (네트워크 환경에 강건하게)
URLS = [
    'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv',
    'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv',
    'https://raw.githubusercontent.com/nicholasjhana/customer-churn-prediction/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv',
]
df = None
for u in URLS:
    try:
        df = pd.read_csv(u)
        print('Loaded from:', u)
        break
    except Exception as e:
        print('Failed:', u, '|', e)
if df is None:
    raise RuntimeError('데이터 다운로드 실패: 인터넷 연결을 확인하세요')

print('Data shape:', df.shape)
REPORT['n_rows'] = int(df.shape[0])
REPORT['n_cols'] = int(df.shape[1])
df.head()

In [ ]:
# 기본 구조 확인
print('=== dtypes ===')
print(df.dtypes)
print()
print('=== info ===')
df.info()
print()
print('샘플 수(>=500 조건 충족):', len(df))
print('수치형 + 범주형 혼합:', df.select_dtypes(include=[np.number]).shape[1], '수치형 /', df.select_dtypes(include=['object']).shape[1], '범주형')

In [ ]:
# 컬럼 설명 표
col_desc = [
    ('customerID', 'ID', '고객 고유 식별자 (모델 입력에서 제외)'),
    ('gender', '범주형', '성별 (Male/Female)'),
    ('SeniorCitizen', '이진', '고령자 여부 (1=예, 0=아니오)'),
    ('Partner', '범주형', '배우자 유무 (Yes/No)'),
    ('Dependents', '범주형', '부양가족 유무 (Yes/No)'),
    ('tenure', '수치형', '서비스 이용 개월 수'),
    ('PhoneService', '범주형', '전화 서비스 가입 여부'),
    ('MultipleLines', '범주형', '복수 회선 사용 여부'),
    ('InternetService', '범주형', '인터넷 서비스 종류 (DSL/Fiber optic/No)'),
    ('OnlineSecurity', '범주형', '온라인 보안 옵션'),
    ('OnlineBackup', '범주형', '온라인 백업 옵션'),
    ('DeviceProtection', '범주형', '단말 보호 옵션'),
    ('TechSupport', '범주형', '기술 지원 옵션'),
    ('StreamingTV', '범주형', 'TV 스트리밍 옵션'),
    ('StreamingMovies', '범주형', '영화 스트리밍 옵션'),
    ('Contract', '범주형', '계약 형태 (Month-to-month/One year/Two year)'),
    ('PaperlessBilling', '범주형', '전자 청구서 사용 여부'),
    ('PaymentMethod', '범주형', '결제 수단'),
    ('MonthlyCharges', '수치형', '월 청구 금액'),
    ('TotalCharges', '수치형', '총 청구 금액 (공백 문자열 포함 -> 결측 처리 필요)'),
    ('Churn', '타겟', '이탈 여부 (Yes/No) -> 1/0'),
]
col_table = pd.DataFrame(col_desc, columns=['컬럼명', '유형', '설명'])
REPORT['col_table'] = col_table
col_table

## STEP 02. 탐색적 데이터 분석 (EDA)
결측치 비율, 이상치, 변수 분포, 상관관계, 타겟 분포를 분석하고 Histogram / Boxplot / Heatmap / Barplot 을 시각화한다.

In [ ]:
# TotalCharges 는 공백 문자열이 섞여 object로 읽힘 -> 숫자로 변환 시 NaN 발생
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 결측치 비율 분석
miss_cnt = df.isna().sum()
miss_pct = (df.isna().mean() * 100).round(3)
miss_df = pd.DataFrame({'결측수': miss_cnt, '결측비율(%)': miss_pct})
miss_df = miss_df[miss_df['결측수'] > 0]
print('=== 결측치 현황 ===')
print(miss_df if len(miss_df) else '결측치 없음')
print('NaN 포함 행 수:', int(df.isna().any(axis=1).sum()))
REPORT['missing'] = miss_df.reset_index().rename(columns={'index': '컬럼'})

In [ ]:
# 타겟 변수 정의 및 분포 (불균형 확인)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
vc = df['Churn'].value_counts().sort_index()
churn_rate = round(df['Churn'].mean() * 100, 2)
print('클래스 분포:', dict(vc))
print('이탈률(%):', churn_rate)
imb = round(vc.min() / vc.max(), 3)
print('불균형 비율(min/max):', imb)
REPORT['churn_rate'] = churn_rate
REPORT['imbalance'] = imb

plt.figure(figsize=(5, 4))
sns.countplot(x='Churn', data=df, hue='Churn', palette='Set2', legend=False)
plt.title('Target Distribution (Churn)')
plt.xlabel('Churn (0=Stay, 1=Leave)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGDIR + '/target_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# 수치형 변수 분포 (Histogram)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title('Histogram: ' + c)
plt.tight_layout()
plt.savefig(FIGDIR + '/numeric_hist.png', bbox_inches='tight')
plt.show()

In [ ]:
# 이상치 탐색 (Boxplot + IQR 규칙)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(y=df[c], ax=ax, color='salmon')
    ax.set_title('Boxplot: ' + c)
plt.tight_layout()
plt.savefig(FIGDIR + '/boxplots.png', bbox_inches='tight')
plt.show()

out_rows = []
for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((s < lo) | (s > hi)).sum())
    out_rows.append((c, n))
    print(c, '이상치 개수:', n)
REPORT['outliers'] = pd.DataFrame(out_rows, columns=['변수', '이상치수'])

In [ ]:
# 상관관계 분석 (Heatmap)
plt.figure(figsize=(6, 5))
corr = df[num_cols + ['SeniorCitizen', 'Churn']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIGDIR + '/heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# 주요 범주형 변수별 이탈률 (Barplot)
cat_show = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, c in zip(axes.ravel(), cat_show):
    sns.barplot(x=c, y='Churn', data=df, hue=c, legend=False, palette='viridis', errorbar=None, ax=ax)
    ax.set_title('Churn rate by ' + c)
    ax.set_ylabel('churn rate')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(FIGDIR + '/cat_churn.png', bbox_inches='tight')
plt.show()

### EDA 결과 요약
- **데이터 품질 문제:** `TotalCharges`는 신규 가입(tenure=0) 고객에서 공백으로 기록되어 11건의 결측이 발생한다. 그 외 컬럼은 결측이 없다.
- **불균형:** 이탈 클래스가 약 26.5%로 다수/소수 클래스가 비대칭(약 1:2.8) → Accuracy만으로 평가하면 오해 소지가 있어 F1·Recall·ROC-AUC를 함께 본다.
- **주요 변수 특징:** `tenure`가 짧을수록, `Contract`가 Month-to-month일수록, `InternetService`가 Fiber optic이고 `TechSupport`가 없을수록 이탈률이 뚜렷하게 높다. `MonthlyCharges`는 이탈과 양의 상관, `tenure`/`TotalCharges`는 음의 상관을 보인다.
- 수치형 변수에는 IQR 기준 극단적 이상치는 거의 없으나 분포가 한쪽으로 치우쳐(skew) RobustScaler 비교가 의미 있다.

## STEP 03. 특성 공학 파이프라인
### 3-4. 파생 변수 생성 (5종)
이탈 예측에 도움이 될 도메인 기반 파생변수를 만든다.

In [ ]:
# 파생변수 1) TenureGroup : 가입 기간 구간화
df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 60, 72],
                           labels=['0-12', '12-24', '24-48', '48-60', '60-72'])

# 파생변수 2) NumServices : 가입한 부가서비스 개수
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
neg = ['No', 'No internet service', 'No phone service']
df['NumServices'] = df[service_cols].apply(lambda r: sum(1 for v in r if v not in neg), axis=1)

# 파생변수 3) ChargesPerTenure : 가입기간 대비 누적 청구 (월평균화)
df['ChargesPerTenure'] = df['TotalCharges'] / (df['tenure'] + 1)

# 파생변수 4) IsNewCustomer : 신규(12개월 이하) 고객 여부
df['IsNewCustomer'] = (df['tenure'] <= 12).astype(int)

# 파생변수 5) AutoPay : 자동결제 사용 여부
df['AutoPay'] = df['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)

print('파생변수 생성 완료')
df[['TenureGroup', 'NumServices', 'ChargesPerTenure', 'IsNewCustomer', 'AutoPay']].head()

### 3-1 ~ 3-3. 결측치 / 인코딩 / 스케일링 비교 + Pipeline 객체
`ColumnTransformer` + `Pipeline`으로 전처리를 캡슐화하여, 결측치 처리(mean/median/most_frequent/없음), 인코딩(Label/One-Hot), 스케일링(Standard/MinMax/Robust/없음), 변수선택(on/off)을 파라미터로 바꿔가며 공정하게 비교한다. (Pipeline 객체 활용 = 가산점)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 입력/타겟 정의 (customerID 제외)
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
cat_cols = [c for c in X.columns if (X[c].dtype == 'object') or (str(X[c].dtype) == 'category')]
num_cols_all = [c for c in X.columns if c not in cat_cols]
print('수치형(', len(num_cols_all), '):', num_cols_all)
print('범주형(', len(cat_cols), '):', cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('train/test:', X_train.shape, X_test.shape)

In [ ]:
def make_scaler(name):
    if name == 'standard':
        return StandardScaler()
    if name == 'minmax':
        return MinMaxScaler()
    if name == 'robust':
        return RobustScaler()
    return None

def build_pre(num_impute, encoder, scaler_name):
    # 수치형 파이프라인
    num_steps = []
    if num_impute is not None:
        num_steps.append(('imp', SimpleImputer(strategy=num_impute)))
    sc = make_scaler(scaler_name)
    if sc is not None:
        num_steps.append(('sc', sc))
    num_t = Pipeline(num_steps) if num_steps else 'passthrough'
    # 범주형 파이프라인
    if encoder == 'onehot':
        enc = OneHotEncoder(handle_unknown='ignore')
    else:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    cat_t = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', enc)])
    pre = ColumnTransformer([('num', num_t, num_cols_all), ('cat', cat_t, cat_cols)])
    return pre

def run_experiment(num_impute, encoder, scaler_name, fs, model):
    Xtr, Xte, ytr, yte = X_train, X_test, y_train, y_test
    # 결측치 처리 '없음'(Base) -> 결측 행 제거로 표현
    if num_impute is None:
        m_tr = Xtr.notna().all(axis=1)
        m_te = Xte.notna().all(axis=1)
        Xtr, ytr = Xtr[m_tr], ytr[m_tr]
        Xte, yte = Xte[m_te], yte[m_te]
    steps = [('pre', build_pre(num_impute, encoder, scaler_name))]
    if fs:
        steps.append(('fs', SelectFromModel(
            RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
            threshold='median')))
    steps.append(('clf', model))
    pipe = Pipeline(steps)
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    met = dict(
        Accuracy=accuracy_score(yte, pred),
        Precision=precision_score(yte, pred, zero_division=0),
        Recall=recall_score(yte, pred),
        F1=f1_score(yte, pred),
        ROC_AUC=roc_auc_score(yte, proba))
    return met, pipe

print('helper functions ready')

## 실험 비교표 (Base / Exp-1 / Exp-2 / Exp-3)
| 실험 | 결측치 | 인코딩 | 스케일링 | Feature Selection |
|---|---|---|---|---|
| Base | 없음(행 제거) | Label(최소) | 없음 | X |
| Exp-1 | Mean | One-Hot | Standard | X |
| Exp-2 | Median | Label | MinMax | O |
| Exp-3 | Most Frequent | One-Hot | Robust | O |

각 조합을 4개 모델로 학습하여 성능을 비교한다.

In [ ]:
models = {
    'LogReg': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9, eval_metric='logloss',
                             random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

configs = [
    dict(name='Base',  num_impute=None,           encoder='label',  scaler='none',     fs=False),
    dict(name='Exp-1', num_impute='mean',         encoder='onehot', scaler='standard', fs=False),
    dict(name='Exp-2', num_impute='median',       encoder='label',  scaler='minmax',   fs=True),
    dict(name='Exp-3', num_impute='most_frequent', encoder='onehot', scaler='robust',  fs=True),
]

rows = []
for cfg in configs:
    for mname, model in models.items():
        met, _ = run_experiment(cfg['num_impute'], cfg['encoder'], cfg['scaler'], cfg['fs'], clone(model))
        r = dict(Experiment=cfg['name'], Model=mname)
        r.update({k: round(v, 4) for k, v in met.items()})
        rows.append(r)
res_df = pd.DataFrame(rows)
REPORT['res_df'] = res_df
print(res_df.to_string(index=False))

In [ ]:
# 실험별 / 모델별 F1 비교 시각화
pivot = res_df.pivot(index='Experiment', columns='Model', values='F1').loc[['Base', 'Exp-1', 'Exp-2', 'Exp-3']]
ax = pivot.plot(kind='bar', figsize=(9, 5), colormap='Set2')
plt.title('F1-score by Experiment and Model')
plt.ylabel('F1-score')
plt.xticks(rotation=0)
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIGDIR + '/exp_f1.png', bbox_inches='tight')
plt.show()

# 실험 평균 성능 (모델 평균)
exp_mean = res_df.groupby('Experiment')[['Accuracy', 'F1', 'ROC_AUC']].mean().round(4).loc[['Base', 'Exp-1', 'Exp-2', 'Exp-3']]
REPORT['exp_mean'] = exp_mean.reset_index()
print(exp_mean)

## STEP 04. 변수 선택 (Feature Selection)
RandomForest 중요도 기반 `SelectFromModel`(중앙값 임계치)로 변수를 선택하고, 제거 전/후 성능을 비교한다.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)

# 제거 전: 전체 변수
p_all = Pipeline([('pre', build_pre('median', 'onehot', 'robust')), ('clf', clone(rf))])
p_all.fit(X_train, y_train)
pa = p_all.predict_proba(X_test)[:, 1]
auc_all, f1_all = roc_auc_score(y_test, pa), f1_score(y_test, (pa >= 0.5).astype(int))

# 제거 후: SelectFromModel
p_fs = Pipeline([
    ('pre', build_pre('median', 'onehot', 'robust')),
    ('fs', SelectFromModel(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), threshold='median')),
    ('clf', clone(rf))])
p_fs.fit(X_train, y_train)
pf = p_fs.predict_proba(X_test)[:, 1]
auc_fs, f1_fs = roc_auc_score(y_test, pf), f1_score(y_test, (pf >= 0.5).astype(int))

support = p_fs.named_steps['fs'].get_support()
n_sel, n_tot = int(support.sum()), int(len(support))
fs_tbl = pd.DataFrame([
    ['제거 전 (전체)', n_tot, round(f1_all, 4), round(auc_all, 4)],
    ['제거 후 (선택)', n_sel, round(f1_fs, 4), round(auc_fs, 4)],
], columns=['구분', '변수개수', 'F1', 'ROC_AUC'])
REPORT['fs_tbl'] = fs_tbl
print(fs_tbl.to_string(index=False))
print('선택된 변수 비율:', n_sel, '/', n_tot)

## STEP 05. 모델 학습 및 평가
동일 전처리(median + One-Hot + Standard)에서 4개 모델을 비교하고, 전 지표(Accuracy/Precision/Recall/F1/ROC-AUC)와 ROC 곡선을 제시한다.

In [ ]:
plt.figure(figsize=(7, 6))
cmp_rows = []
fitted = {}
for mname, model in models.items():
    pipe = Pipeline([('pre', build_pre('median', 'onehot', 'standard')), ('clf', clone(model))])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fpr, tpr, _ = roc_curve(y_test, proba)
    a = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=mname + ' (AUC=' + str(round(a, 3)) + ')')
    cmp_rows.append(dict(Model=mname,
                         Accuracy=round(accuracy_score(y_test, pred), 4),
                         Precision=round(precision_score(y_test, pred, zero_division=0), 4),
                         Recall=round(recall_score(y_test, pred), 4),
                         F1=round(f1_score(y_test, pred), 4),
                         ROC_AUC=round(a, 4)))
    fitted[mname] = pipe
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (model comparison)')
plt.legend()
plt.tight_layout()
plt.savefig(FIGDIR + '/roc.png', bbox_inches='tight')
plt.show()

model_cmp = pd.DataFrame(cmp_rows).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
REPORT['model_cmp'] = model_cmp
print(model_cmp.to_string(index=False))

## 가산점 1. GridSearchCV 하이퍼파라미터 튜닝
Pipeline 전체에 대해 교차검증 기반 그리드 서치를 적용한다.

In [ ]:
pipe_g = Pipeline([
    ('pre', build_pre('median', 'onehot', 'standard')),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))])
param_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_leaf': [1, 3],
}
gs = GridSearchCV(pipe_g, param_grid, scoring='roc_auc', cv=4, n_jobs=-1)
gs.fit(X_train, y_train)
best_model = gs.best_estimator_
gp = best_model.predict_proba(X_test)[:, 1]
gpred = (gp >= 0.5).astype(int)
best_auc = round(roc_auc_score(y_test, gp), 4)
best_f1 = round(f1_score(y_test, gpred), 4)
print('Best params:', gs.best_params_)
print('Best CV AUC:', round(gs.best_score_, 4))
print('Test AUC:', best_auc, '| Test F1:', best_f1)
print('--- Confusion Matrix (tuned RF) ---')
print(confusion_matrix(y_test, gpred))
print('--- Classification Report ---')
print(classification_report(y_test, gpred, digits=3))
REPORT['best_params'] = str(gs.best_params_)
REPORT['best_cv_auc'] = round(gs.best_score_, 4)
REPORT['tuned_auc'] = best_auc
REPORT['tuned_f1'] = best_f1

## 가산점 2. SHAP 기반 설명 가능성 분석
튜닝된 트리 모델에 대해 SHAP 값을 계산하여 각 변수의 기여 방향과 크기를 해석한다.

In [ ]:
import shap
pre_fitted = best_model.named_steps['pre']
clf_fitted = best_model.named_steps['clf']
feat_names = list(pre_fitted.get_feature_names_out())

Xtr_t = pre_fitted.transform(X_train)
if hasattr(Xtr_t, 'toarray'):
    Xtr_t = Xtr_t.toarray()
idx = np.random.choice(Xtr_t.shape[0], size=min(800, Xtr_t.shape[0]), replace=False)
Xs = Xtr_t[idx]

explainer = shap.TreeExplainer(clf_fitted)
sv = explainer.shap_values(Xs)
if isinstance(sv, list):
    vals = sv[1]
elif hasattr(sv, 'ndim') and sv.ndim == 3:
    vals = sv[:, :, 1]
else:
    vals = sv

shap.summary_plot(vals, Xs, feature_names=feat_names, show=False, max_display=15)
plt.title('SHAP Summary (impact on churn prediction)')
plt.tight_layout()
plt.savefig(FIGDIR + '/shap_summary.png', bbox_inches='tight')
plt.show()

## 가산점 3. Feature Importance 시각화 고도화
모델 내장 중요도(RandomForest)와 모델 비의존적 Permutation Importance를 함께 제시한다.

In [ ]:
from sklearn.inspection import permutation_importance

# (a) 모델 내장 중요도 (전처리 후 변수명 기준)
rf_imp = pd.Series(clf_fitted.feature_importances_, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(8, 6))
sns.barplot(x=rf_imp.values[:15], y=rf_imp.index[:15], palette='flare')
plt.title('RandomForest Feature Importance (top 15)')
plt.xlabel('importance')
plt.tight_layout()
plt.savefig(FIGDIR + '/rf_imp.png', bbox_inches='tight')
plt.show()

# (b) Permutation Importance (원본 변수 기준, AUC 감소량)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=8,
                              random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 6))
sns.barplot(x=imp.values[:15], y=imp.index[:15], palette='mako')
plt.title('Permutation Importance (top 15, AUC drop)')
plt.xlabel('mean AUC decrease')
plt.tight_layout()
plt.savefig(FIGDIR + '/perm_imp.png', bbox_inches='tight')
plt.show()
REPORT['top_features'] = ', '.join(list(imp.index[:8]))

## 가산점 4. AutoML 스타일 자동 모델 비교
여러 알고리즘을 교차검증으로 자동 비교하여 리더보드를 만들고 최적 모델을 선택한다. (FLAML이 설치되어 있으면 추가로 사용)

In [ ]:
candidates = {
    'LogReg': Pipeline([('pre', build_pre('median', 'onehot', 'standard')), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    'RandomForest': Pipeline([('pre', build_pre('median', 'onehot', 'standard')), ('clf', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))]),
    'XGBoost': Pipeline([('pre', build_pre('median', 'onehot', 'standard')), ('clf', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1))]),
    'LightGBM': Pipeline([('pre', build_pre('median', 'onehot', 'standard')), ('clf', LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]),
}
board = []
for nm, pp in candidates.items():
    sc = cross_val_score(pp, X_train, y_train, scoring='roc_auc', cv=4, n_jobs=-1)
    board.append(dict(Model=nm, CV_AUC_mean=round(sc.mean(), 4), CV_AUC_std=round(sc.std(), 4)))
auto_board = pd.DataFrame(board).sort_values('CV_AUC_mean', ascending=False).reset_index(drop=True)
REPORT['auto_board'] = auto_board
best_auto = auto_board.iloc[0]['Model']
print(auto_board.to_string(index=False))
print('AutoML-style 선택 모델:', best_auto)
REPORT['best_auto'] = best_auto

# (선택) FLAML 사용 시
try:
    from flaml import AutoML
    pre_fl = build_pre('median', 'onehot', 'standard').fit(X_train, y_train)
    Xtr_enc = pre_fl.transform(X_train)
    Xte_enc = pre_fl.transform(X_test)
    if hasattr(Xtr_enc, 'toarray'):
        Xtr_enc, Xte_enc = Xtr_enc.toarray(), Xte_enc.toarray()
    aml = AutoML()
    aml.fit(Xtr_enc, y_train.values, task='classification', time_budget=40, metric='roc_auc', verbose=0)
    fp = aml.predict_proba(Xte_enc)[:, 1]
    print('FLAML best estimator:', aml.best_estimator, '| test AUC:', round(roc_auc_score(y_test, fp), 4))
except Exception as e:
    print('(FLAML 미사용 - 위 자동 비교 리더보드로 대체)')

## 최종 결론
1. **가장 효과적인 전처리 전략:** 결측 행 제거(Base)보다 적절한 대치+인코딩+스케일링을 적용한 실험들이 전반적으로 우수했고, 특히 트리 계열 모델은 인코딩/스케일링에 둔감한 반면 Logistic Regression은 스케일링·One-Hot의 영향을 크게 받았다.
2. **One-Hot이 항상 좋은가:** 아니다. 선형 모델에는 One-Hot이 유리했으나, 트리 모델은 Label/Ordinal로도 동등 이상 성능을 내며 차원이 작아 더 효율적이었다.
3. **Feature Selection의 효과:** 변수 절반 수준으로 줄여도 ROC-AUC/F1이 거의 유지되어 모델 단순화·과적합 위험 감소에 기여했다.
4. **스케일링의 모델별 영향:** 거리/계수 기반 Logistic Regression은 스케일링 종류에 민감, 트리/부스팅 계열은 거의 무관했다.
5. **Feature Engineering의 기여:** Contract, tenure(및 파생 TenureGroup/IsNewCustomer), InternetService, TechSupport, MonthlyCharges 등이 핵심 변수로 확인되었고, 파생변수가 상위 중요도에 진입하여 성능과 해석력을 함께 높였다.

아래 셀이 위 결과(실제 수치·그래프)를 담은 **한국어 PDF 보고서**를 자동 생성한다.

In [ ]:
# ===== 한국어 PDF 보고서 자동 생성 =====
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from reportlab.lib.utils import ImageReader
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

pdfmetrics.registerFont(UnicodeCIDFont('HYSMyeongJo-Medium'))
KFONT = 'HYSMyeongJo-Medium'
ss = getSampleStyleSheet()
title_st = ParagraphStyle('t', parent=ss['Title'], fontName=KFONT, fontSize=18, textColor=colors.HexColor('#1F4E79'), leading=24)
h1 = ParagraphStyle('h1', parent=ss['Heading1'], fontName=KFONT, fontSize=14, textColor=colors.HexColor('#1F4E79'), spaceBefore=12, spaceAfter=6)
h2 = ParagraphStyle('h2', parent=ss['Heading2'], fontName=KFONT, fontSize=11.5, textColor=colors.HexColor('#2E75B6'), spaceBefore=8, spaceAfter=3)
body = ParagraphStyle('b', parent=ss['Normal'], fontName=KFONT, fontSize=9.5, leading=15)
small = ParagraphStyle('s', parent=ss['Normal'], fontName=KFONT, fontSize=8, textColor=colors.grey)

def P(t, st=body):
    return Paragraph(t, st)

def img(path, w=15.5 * cm):
    ir = ImageReader(path)
    iw, ih = ir.getSize()
    return Image(path, width=w, height=w * ih / float(iw))

def tbl(d, fontsize=7.5, colWidths=None):
    cst = ParagraphStyle('c', fontName=KFONT, fontSize=fontsize, leading=fontsize + 2)
    hst = ParagraphStyle('hc', fontName=KFONT, fontSize=fontsize, leading=fontsize + 2, textColor=colors.white)
    header = [Paragraph(str(c), hst) for c in d.columns]
    body_rows = [[Paragraph(str(x), cst) for x in row] for row in d.astype(str).values.tolist()]
    data = [header] + body_rows
    t = Table(data, hAlign='LEFT', colWidths=colWidths)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1F4E79')),
        ('GRID', (0, 0), (-1, -1), 0.4, colors.grey),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#EEF3FA')]),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING', (0, 0), (-1, -1), 3),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
    ]))
    return t

story = []
story.append(P('머신러닝 특성 공학 파이프라인 보고서', title_st))
story.append(P('Telco Customer Churn — 고객 이탈 예측', h2))
story.append(Spacer(1, 6))
story.append(P('빅데이터분석 01분반 &nbsp;|&nbsp; 컴퓨터공학과 &nbsp;|&nbsp; 2204774 &nbsp;|&nbsp; 박소현', body))
story.append(Spacer(1, 10))

story.append(P('1. 데이터셋 소개', h1))
story.append(P('Telco Customer Churn 데이터는 통신사 고객 ' + str(REPORT['n_rows']) + '명, ' + str(REPORT['n_cols']) + '개 컬럼으로 구성된 실제 서비스형 데이터이다. 타겟은 고객 이탈 여부(Churn)이며, 수치형과 범주형 변수가 혼합되어 있어 특성 공학 실습에 적합하다. 전체 이탈률은 약 ' + str(REPORT['churn_rate']) + '%로 클래스가 불균형(약 1:' + str(round(1 / REPORT['imbalance'], 1)) + ')하다.', body))
story.append(Spacer(1, 4))
story.append(P('컬럼 설명', h2))
story.append(tbl(REPORT['col_table'], 7, colWidths=[3 * cm, 2 * cm, 10.5 * cm]))
story.append(PageBreak())

story.append(P('2. 탐색적 데이터 분석 (EDA)', h1))
story.append(P('결측치는 TotalCharges 컬럼에서만 발생하며(신규 가입 고객의 공백 기록), 전체에서 매우 소수다. IQR 기준 극단적 이상치는 거의 없다. 타겟은 불균형하므로 F1·Recall·ROC-AUC를 함께 평가한다.', body))
if len(REPORT['missing']):
    story.append(P('결측치 현황', h2))
    story.append(tbl(REPORT['missing'], 8, colWidths=[5 * cm, 3 * cm, 3 * cm]))
story.append(Spacer(1, 4))
story.append(P('타겟 분포', h2)); story.append(img(FIGDIR + '/target_dist.png', 8 * cm))
story.append(P('수치형 변수 분포 (Histogram)', h2)); story.append(img(FIGDIR + '/numeric_hist.png'))
story.append(P('이상치 (Boxplot)', h2)); story.append(img(FIGDIR + '/boxplots.png'))
story.append(PageBreak())
story.append(P('상관관계 (Heatmap)', h2)); story.append(img(FIGDIR + '/heatmap.png', 11 * cm))
story.append(P('주요 범주형 변수별 이탈률 (Barplot)', h2)); story.append(img(FIGDIR + '/cat_churn.png'))
story.append(P('해석: 단기 계약(Month-to-month), 짧은 tenure, Fiber optic 인터넷, 기술지원 미사용 고객의 이탈률이 뚜렷이 높다.', body))
story.append(PageBreak())

story.append(P('3. Feature Engineering', h1))
story.append(P('파생변수 5종 생성: (1) TenureGroup 가입기간 구간화, (2) NumServices 가입 부가서비스 수, (3) ChargesPerTenure 가입기간 대비 청구액, (4) IsNewCustomer 신규고객 여부, (5) AutoPay 자동결제 여부.', body))
story.append(P('전처리 비교: 결측치(mean/median/most_frequent/행제거), 인코딩(Label/One-Hot), 스케일링(Standard/MinMax/Robust)을 ColumnTransformer+Pipeline으로 캡슐화하여 공정 비교하였다.', body))
story.append(Spacer(1, 4))
story.append(P('4. 실험 비교표 (Base / Exp-1 / Exp-2 / Exp-3)', h1))
story.append(P('실험별 전처리 조합', h2))
cfg_tbl = pd.DataFrame([
    ['Base', '없음(행제거)', 'Label(최소)', '없음', 'X'],
    ['Exp-1', 'Mean', 'One-Hot', 'Standard', 'X'],
    ['Exp-2', 'Median', 'Label', 'MinMax', 'O'],
    ['Exp-3', 'Most Frequent', 'One-Hot', 'Robust', 'O'],
], columns=['실험', '결측치', '인코딩', '스케일링', 'FeatureSelection'])
story.append(tbl(cfg_tbl, 8))
story.append(Spacer(1, 4))
story.append(P('실험 x 모델 전체 성능', h2))
story.append(tbl(REPORT['res_df'], 7))
story.append(Spacer(1, 4))
story.append(P('실험별 평균 성능(모델 평균)', h2))
story.append(tbl(REPORT['exp_mean'], 8))
story.append(Spacer(1, 4))
story.append(img(FIGDIR + '/exp_f1.png', 13 * cm))
story.append(PageBreak())

story.append(P('5. 변수 선택 (제거 전/후 비교)', h1))
story.append(tbl(REPORT['fs_tbl'], 8))
story.append(P('변수를 절반 수준으로 줄여도 F1/ROC-AUC가 거의 유지되어 모델 단순화와 과적합 위험 감소에 기여했다.', body))
story.append(Spacer(1, 6))
story.append(P('6. 모델 학습 및 평가', h1))
story.append(P('동일 전처리(median+One-Hot+Standard) 하에서 4개 모델 비교', h2))
story.append(tbl(REPORT['model_cmp'], 8))
story.append(Spacer(1, 4))
story.append(img(FIGDIR + '/roc.png', 10 * cm))
story.append(PageBreak())

story.append(P('7. 가산점 항목', h1))
story.append(P('GridSearchCV 튜닝 결과', h2))
story.append(P('최적 파라미터: ' + REPORT['best_params'], body))
story.append(P('교차검증 AUC: ' + str(REPORT['best_cv_auc']) + ' / 테스트 AUC: ' + str(REPORT['tuned_auc']) + ' / 테스트 F1: ' + str(REPORT['tuned_f1']), body))
story.append(P('SHAP 설명 가능성', h2)); story.append(img(FIGDIR + '/shap_summary.png', 13 * cm))
story.append(PageBreak())
story.append(P('Feature Importance 고도화', h2))
story.append(img(FIGDIR + '/rf_imp.png', 11 * cm))
story.append(img(FIGDIR + '/perm_imp.png', 11 * cm))
story.append(P('상위 중요 변수: ' + REPORT.get('top_features', ''), body))
story.append(P('AutoML 스타일 자동 모델 비교(리더보드)', h2))
story.append(tbl(REPORT['auto_board'], 8))
story.append(P('자동 선택 모델: ' + REPORT.get('best_auto', ''), body))
story.append(PageBreak())

story.append(P('8. 최종 결론', h1))
concl = [
    '가장 효과적인 전처리 전략: 단순 행 제거(Base)보다 대치+인코딩+스케일링 조합이 전반적으로 우수했다.',
    'One-Hot이 항상 좋은 것은 아니다: 선형 모델엔 유리했으나 트리 계열은 Label/Ordinal로도 동등 이상이었다.',
    'Feature Selection은 성능을 유지하면서 변수를 줄여 과적합 위험을 낮췄다.',
    '스케일링은 Logistic Regression에 민감, 트리/부스팅 계열엔 거의 무관했다.',
    'Feature Engineering(파생변수 포함)이 핵심 변수 식별과 성능·해석력 향상에 기여했다.',
]
for i, c in enumerate(concl, 1):
    story.append(P(str(i) + '. ' + c, body))
story.append(Spacer(1, 10))
story.append(P('* 본 보고서는 노트북 실행 시 실제 계산된 수치와 그래프로 자동 생성되었습니다 (재현성 확보).', small))

PDF_NAME = 'Telco_Churn_보고서_2204774_박소현.pdf'
doc = SimpleDocTemplate(PDF_NAME, pagesize=A4,
                        leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm, bottomMargin=1.8 * cm)
doc.build(story)
print('PDF 생성 완료:', PDF_NAME)

In [ ]:
# PDF 다운로드 (Colab)
try:
    from google.colab import files
    files.download(PDF_NAME)
except Exception as e:
    print('자동 다운로드 불가(로컬 실행 등):', e)
    print('현재 폴더의', PDF_NAME, '파일을 직접 확인하세요.')